# External bridges trade-off (coverage vs ambiguity vs runtime)

This notebook quantifies a core IDTrack design claim:

- External namespaces can **increase connectivity** (reduce 1→0), but they can also **amplify ambiguity** (increase 1→n) and runtime.
- IDTrack treats external inclusion as a **reviewable contract** (YAML allowlist) rather than a hidden default.

## Rationale

This experiment is a marketing-friendly way to justify why IDTrack uses a curated external allowlist:

- **Curated externals** are a controlled bridge that can reconnect otherwise disconnected Ensembl histories.
- **Unbounded externals** often add highly connected identifiers that explode transitive ambiguity, increasing 1→n and cost.

## What to report (Methods-facing knobs)

- Snapshot release (graph boundary)
- YAML allowlist (external contract)
- Strategy (`best` vs `all`) for ambiguity exposure

We compare two graph configurations for the same organism/snapshot:

1. **Curated externals** (`narrow_external=True`): uses the configured YAML to include only selected external databases.
2. **All externals** (`narrow_external=False`): includes all external xref tables available in Ensembl (often much noisier).

Outputs:
- Figure: `idtrack-manuscript/figures/fig_external_bridges_tradeoff.pdf`

Caching:
- Results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/external_bridges_tradeoff/`.
- If caches are missing, the notebook computes them (graph loading can be memory-intensive).

## Interpretation guide

- Panel 1: how the outcome profile shifts (coverage vs ambiguity).
- Panel 2: throughput impact of widening external scope.
- Panel 3: the measured expansion of external scope (DBs/nodes).


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_figures_dir,
    read_pickle,
    write_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='external_bridges_tradeoff')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISM_ALIAS = 'human'
SNAPSHOT_RELEASE = 114
TO_RELEASE = 107
FINAL_DATABASE = 'HGNC Symbol'
STRATEGY = 'all'

N_QUERIES = 500
RANDOM_SEED = 0

RESULTS_PKL = CACHE_DIR / (
    f"tradeoff_{ORGANISM_ALIAS}_snapshot{SNAPSHOT_RELEASE}_to{TO_RELEASE}_final{FINAL_DATABASE}_"
    f"strategy{STRATEGY}_n{N_QUERIES}_seed{RANDOM_SEED}.pickle"
)

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import numpy as np

if RESULTS_PKL.exists():
    payload = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
else:
    import idtrack
    from idtrack import DB
    from idtrack._track import Track

    rng = np.random.default_rng(RANDOM_SEED)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = int(SNAPSHOT_RELEASE)
    if snapshot > int(latest):
        raise ValueError(f'snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}')

    # Build a DatabaseManager once (shared across Track instances)
    dm = api.get_database_manager(organism_name=organism, snapshot_release=snapshot)

    def build_track(narrow_external: bool) -> Track:
        return Track(dm, narrow=True, narrow_external=narrow_external)

    def reservoir_sample(nodes, prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    # Use the curated graph to pick queries (ENS nodes are the same across modes)
    t_curated = build_track(narrow_external=True)
    api.track = t_curated

    queries = reservoir_sample(api.track.graph.nodes, 'ENSG', N_QUERIES)
    if not queries:
        raise RuntimeError('No ENSG nodes found; cannot run trade-off experiment.')

    def graph_stats(track: Track) -> dict:
        g = track.graph
        n_nodes = int(g.number_of_nodes())
        n_edges = int(g.number_of_edges())
        n_ext_dbs = int(len(getattr(g, 'available_external_databases', [])))
        n_ext_nodes = 0
        for _n, att in g.nodes(data=True):
            if att.get(DB.node_type_str) == DB.nts_external:
                n_ext_nodes += 1
        return {
            'n_nodes': n_nodes,
            'n_edges': n_edges,
            'n_external_dbs': n_ext_dbs,
            'n_external_nodes': int(n_ext_nodes),
        }

    def run_mode(narrow_external: bool) -> dict:
        track = build_track(narrow_external=narrow_external)
        api.track = track

        stats = graph_stats(track)

        t0 = time.perf_counter()
        matchings = api.convert_identifier_multiple(
            queries,
            to_release=int(TO_RELEASE),
            final_database=FINAL_DATABASE,
            strategy=STRATEGY,
            verbose=True,
            pbar_prefix=('curated' if narrow_external else 'extall'),
            explain=False,
        )
        dt = time.perf_counter() - t0

        bins = api.classify_multiple_conversion(matchings)
        n = len(bins['input_identifiers'])

        one0 = len(bins['matching_1_to_0'])
        one1 = len(bins['matching_1_to_1'])
        onen = len(bins['matching_1_to_n'])

        return {
            'mode': 'curated' if narrow_external else 'extall',
            **stats,
            'n_queries': int(n),
            'seconds': float(dt),
            'it_per_s': float(n / dt) if dt else float('nan'),
            'frac_1_to_0': float(one0 / n) if n else float('nan'),
            'frac_1_to_1': float(one1 / n) if n else float('nan'),
            'frac_1_to_n': float(onen / n) if n else float('nan'),
        }

    rows = [run_mode(True), run_mode(False)]
    payload = {
        'params': {
            'organism_alias': ORGANISM_ALIAS,
            'snapshot_release': snapshot,
            'to_release': int(TO_RELEASE),
            'final_database': FINAL_DATABASE,
            'strategy': STRATEGY,
            'n_queries': int(N_QUERIES),
            'random_seed': int(RANDOM_SEED),
        },
        'rows': rows,
    }

    write_pickle(payload, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

payload


In [ ]:
# -------------------- Plot trade-off figure --------------------

rows = pd.DataFrame(payload['rows']).set_index('mode')
rows


In [ ]:
# -------------------- Export figure --------------------

rows = pd.DataFrame(payload['rows']).set_index('mode')

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), constrained_layout=True)
ax0, ax1, ax2 = axes

# Outcome fractions
rows[['frac_1_to_0', 'frac_1_to_1', 'frac_1_to_n']].plot(
    kind='bar',
    stacked=True,
    ax=ax0,
    color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
)
ax0.set_ylim(0, 1)
ax0.set_ylabel('Fraction of queries')
ax0.set_title('Outcome profile')
ax0.legend(['1→0', '1→1', '1→n'], loc='upper right', frameon=True)

# Runtime
rows['it_per_s'].plot(kind='bar', ax=ax1, color=MANUSCRIPT_COLORS['1→1'])
ax1.set_ylabel('Iterations per second (it/s)')
ax1.set_title('Runtime throughput')

# Graph size / external scope
rows[['n_external_dbs', 'n_external_nodes']].plot(kind='bar', ax=ax2)
ax2.set_title('External scope')
ax2.set_ylabel('Count')
ax2.legend(['# external DBs', '# external nodes'], frameon=True)

out_fig = MANUSCRIPT_FIGURES / 'fig_external_bridges_tradeoff.pdf'
fig.savefig(out_fig, bbox_inches='tight')
print('Saved:', out_fig)
